# App evaluation - card faithfulness, retrieval, successor task

Uses the knowledge store built in notebook 15. Outputs in `results/app_eval/`. People do the checking; this notebook makes the sheets and computes the numbers.

1. **Card spot check**: mark 50 cards faithful y/n.
2. **Retrieval**: write a question for each sampled card **in your own words** (not the title) and list every relevant card id; then Recall@k and MRR with and without filters.
3. **Successor task** (RQ4): counterbalanced sessions, blind rating, within-participant comparison. Indicative with a small group.

In [ ]:
# --- Colab setup (skip when running locally) ---
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = "https://github.com/WarunaS/knowledge-retention-ml.git"
    !git clone -q {REPO} /content/knowledge-retention-ml || git -C /content/knowledge-retention-ml pull -q
    %pip install -q -e "/content/knowledge-retention-ml[ml,app]"
    os.environ["KRS_DATA_ROOT"] = "/content/drive/MyDrive/krs/data"
    os.environ["KRS_MODELS_DIR"] = "/content/drive/MyDrive/krs/models"
    os.environ["KRS_RESULTS_DIR"] = "/content/drive/MyDrive/krs/results"

In [ ]:
from krs.app.storage import KnowledgeStore
from krs.evaluate import retrieval_eval as RE

store = KnowledgeStore()
RE.export_card_check(store, 50)     # results/app_eval/card_check.csv
RE.export_query_sheet(store, 50)    # results/app_eval/queries.csv

In [ ]:
RE.card_check_rate()   # after marking the 'faithful' column y/n

In [ ]:
import pandas as pd

rep = RE.evaluate_retrieval(store, RE.load_queries())   # after writing the questions
pd.DataFrame(rep["modes"]).T

In [ ]:
rep["filters_help"], rep["note"]

## Successor task

Pick 8-12 questions from the question sheet about work the participants did not do. Each participant answers every question once: half **with** the system (ask-predecessor, search, handover pack), half **without** (raw logs only). Get consent first; participants are P1, P2, ...

In [ ]:
from krs.evaluate import successor as S

qs = {q.query_id: q.question for q in RE.load_queries()[:10]}
S.export_participant_sheets(qs, ["P1", "P2", "P3", "P4"])   # results/app_eval/successor/

In [ ]:
import glob
from krs.config import RESULTS_DIR

responses = S.read_responses(sorted(glob.glob(str(RESULTS_DIR / "app_eval" / "successor" / "participant_*.csv"))))
S.export_rating_sheet(responses)   # give rating_sheet.csv to the rater; keep rating_key.csv private

In [ ]:
S.analyse()   # after the rater fills rating_1_to_5